# Обучить Агента решать Acrobot-v1, MountainCar-v0, или LunarLander-v2 (одну на выбор) методом DQN. Найти оптимальные гиперпараметры. Сравнить с алгоритмом Deep Cross-Entropy на графиках.

# Libraries

In [1]:
from typing import Optional, Literal
import random
from warnings import warn

from tqdm.auto import tqdm
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Optimizer, Adam, AdamW

# DCEM

In [1]:
# Нахуй оно мне надо

# Deep Q Network

## ReplayBuffer: memory allocation and management

In [2]:
class MemoryBlock: 
    def __init__(self, 
                 state: torch.FloatTensor | np.ndarray, 
                 action: torch.IntTensor | np.ndarray, 
                 reward: torch.FloatTensor | np.ndarray, 
                 next_state: torch.FloatTensor | np.ndarray, 
                 done: torch.IntTensor | np.ndarray,
                 device: str = None
                 ):
        self.state = torch.from_numpy(state) if isinstance(state, np.ndarray) else state
        self.action = torch.tensor(action) if isinstance(state, np.ndarray) else action
        self.reward = torch.tensor(reward) if isinstance(state, np.ndarray) else reward
        self.next_state = torch.from_numpy(next_state) if isinstance(state, np.ndarray) else next_state
        self.done = torch.tensor(done) if isinstance(state, np.ndarray) else done
        
        if device:
            self.to(device)
        self.__check_device()
        
    def to(self, device) -> None:
        self.state = self.state.to(device)
        self.action = self.action.to(device)
        self.reward = self.reward.to(device)
        self.next_state = self.next_state.to(device)
        self.done = self.done.to(device)

    def __str__(self) -> str:
        return (
            f"MemoryBlock(\n"
            f"state={self.state!r}\n"
            f"action={self.action!r}\n"
            f"reward={self.reward!r}\n"
            f"next_state={self.next_state!r}\n"
            f"done={self.done!r}\n)"
        )
    
    def __repr__(self) -> str:
        return (
            f"MemoryBlock("
            f"state={self.state!r}, "
            f"action={self.action!r}, "
            f"reward={self.reward!r}, "
            f"next_state={self.next_state!r}, "
            f"done={self.done!r})"
        )
        
    def __check_device(self) -> None:
        shared_device = {
            self.state.device, self.action.device, self.reward.device, 
            self.next_state.device, self.done.device
        }
        if len(shared_device) != 1:
            raise AssertionError(
                f"All element in memory block must be in the same device, but got: "
                f"{self.state.device=}, {self.action.device=}, {self.reward.device=}, "
                f"{self.next_state.device=}, {self.done.device}"
            )
        
class Memory:
    def __init__(self,
                 buffer_size: int, 
                 state_dim: int, 
                 device: Literal["cpu", "cuda"]
                 ):
        self.device = device
        self.buffer_size = buffer_size
        self.counter = 0
        self._states = torch.empty(self.buffer_size, state_dim, dtype=torch.float32, device=self.device)
        self._actions = torch.empty(self.buffer_size, 1, dtype=torch.int64, device=self.device)
        self._rewards = torch.empty(self.buffer_size, 1, dtype=torch.float32, device=self.device)
        self._next_states = torch.empty(self.buffer_size, state_dim, dtype=torch.float32, device=self.device)
        self._dones = torch.empty(self.buffer_size, 1, dtype=torch.int64, device=self.device)
        
    def __len__(self) -> int:
        return self.counter
    
    def __getitem__(self, index: int) -> MemoryBlock:
        return MemoryBlock(
            self._states[index], 
            self._actions[index], 
            self._rewards[index], 
            self._next_states[index], 
            self._dones[index]
        )
    
    def __setitem__(self, index: int, block: MemoryBlock) -> None:
        block.to(self.device)
        self._states[index] = block.state
        self._actions[index] = block.action
        self._rewards[index] = block.reward
        self._next_states[index] = block.next_state
        self._dones[index] = block.done
        self.counter += 1
    
    def write(self, block: MemoryBlock) -> None:
        index = self.counter % self.buffer_size
        self[index] = block


class ReplayBuffer(Memory):
    def __init__(self, 
                 buffer_size: int, 
                 batch_size: int, 
                 state_dim: int, 
                 device: Literal["cpu", "cuda"]
                 ):
        super().__init__(buffer_size, state_dim, device)
        self.batch_size = batch_size
    
    def sample(self) -> MemoryBlock:
        if len(self) >= self.batch_size:
            indices = torch.randint(
                0, min(len(self), self.buffer_size), 
                (self.batch_size,),
                dtype=torch.int64
            )
            return self[indices]
        warn(
            f"Can't sample MemoryBlock from ReplayBuffer "
            f"before len {len(self)} less then batch_size {self.batch_size}"
        )

## DQN

In [3]:
class DQN(nn.Module):
    def __init__(self, space_dim: int, action_n: int):
        super().__init__()
        self.space_dim = space_dim
        self.action_n = action_n
        self.layers = nn.Sequential(
            nn.Linear(space_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_n)
        )
    
    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        return self.layers(x)


class DQNAgent:
    def __init__(self, 
                 state_dim: int, 
                 action_n: int,
                 buffer_size: int, 
                 batch_size: int,
                 optimizer: Optimizer,
                 lr: Optional[float] = 0.0001,
                 gamma: Optional[float] = 0.98, 
                 eps: Optional[float] = 1.0, 
                 eps_min: Optional[float] = 0.01, 
                 eps_decay: Optional[float] = 0.98, 
                 device: Optional[Literal["cpu", "cuda"]] = None
                 ):
        if device is None:
            warn("No device was defined, device will be selected automatically")
            device = "cuda" if torch.cuda.is_available() else "cpu"
            
        self.device = device 
        self.dqn = DQN(state_dim, action_n).to(self.device)
        self.buffer = ReplayBuffer(buffer_size, batch_size, state_dim, device)
        self.criterion = nn.MSELoss()
        self.optimizer = optimizer(self.dqn.parameters(), lr=lr)
        self.gamma = gamma
        self.eps = eps
        self.eps_min = eps_min
        self.eps_decay = eps_decay

    def get_action(self, state: np.ndarray | torch.FloatTensor) -> int:
        if self.dqn.training and random.uniform(0, 1) < self.eps:
            action = np.random.randint(self.dqn.action_n)
        else:
            if isinstance(state, np.ndarray):
                state = torch.FloatTensor(state)
            with torch.no_grad():
                state = state.to(self.device)
                action = torch.argmax(self.dqn(state).cpu()).item()
        return action
        
    def fit(self, data: MemoryBlock):
        self.buffer.write(data) 
       
        if shuffled_memory := self.buffer.sample(): 
            predicted_q_values = self.dqn.forward(shuffled_memory.state).gather(1, shuffled_memory.action)
            with torch.no_grad():
                max_next_q_values = self.dqn.forward(shuffled_memory.next_state).max(1)[0].unsqueeze(1)
                target_q_values = shuffled_memory.reward + (self.gamma * max_next_q_values * (1 - shuffled_memory.done))
            
            loss = self.criterion(predicted_q_values, target_q_values)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            if self.eps > self.eps_min:
                self.eps *= self.eps_decay

## Utility func

In [4]:
def dqn(
        env: gym.Env, 
        agent: DQNAgent,
        num_trajectories: int, 
        max_trajectory_len: int = 5_000,
        goal: int = 200,
        seed: int = 42
):
    total_rewards = []
    for i in tqdm(range(num_trajectories), desc="Initialize DQN training 🚀"):
        total_reward = 0
    
        state, info = env.reset(seed=seed)
        for _ in range(max_trajectory_len):
            action = agent.get_action(state)
            
            next_state, reward, done, truncated, info = env.step(action)
            total_reward += reward
            
            data = MemoryBlock(
                state, action, reward,
                next_state, done
            )
            agent.fit(data)
            state = next_state
    
            if done:
                break
    
        total_rewards.append(total_reward)
        if len(total_rewards) >= 100 and np.mean(total_rewards[-100:]) >= goal:
            print("Converged 🟩")
            break 
            
        print(f"Total Reward: {total_reward} At {i} trajectory")
    return total_rewards + [total_rewards[-1]] * (num_trajectories - len(total_rewards))

# Setup

## ENV

In [5]:
env = gym.make("LunarLander-v2", render_mode="human")
print(env.observation_space.shape, "\n", env.action_space)

(8,) 
 Discrete(4)


## Constants

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ITERATIONS = 100000

# Train

## DCEM

## DQN

In [7]:
dqn_agent = DQNAgent(
    state_dim=8,
    action_n=4,
    buffer_size=100_000,
    batch_size=1024,
    optimizer=Adam,
    lr=0.0001,
    device=DEVICE
)

dqn(env, dqn_agent, num_trajectories=ITERATIONS)

Initialize DQN training 🚀:   0%|          | 0/100000 [00:00<?, ?it/s]

/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 1 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 2 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 3 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 4 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 5 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 6 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 7 less then batch_size 1024


Total Reward: -214.6634730397425 At 0 trajectory


/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 115 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 116 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 117 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 118 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 119 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 120 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 121 less then ba

Total Reward: -224.0199126058922 At 1 trajectory


...

/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 866 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 867 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 868 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 869 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 870 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 871 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 872 less then ba

Total Reward: -403.638325224108 At 7 trajectory


/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 980 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 981 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 982 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 983 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 984 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 985 less then batch_size 1024
  warn(
/tmp/ipykernel_3102465/632685437.py:118: UserWarning: Can't sample MemoryBlock from ReplayBuffer before len 986 less then ba

Total Reward: -163.4153518452136 At 8 trajectory
Total Reward: -511.6086220899163 At 9 trajectory
Total Reward: -603.0642535598336 At 10 trajectory
Total Reward: -468.73492592874254 At 11 trajectory
Total Reward: -366.28771529484425 At 12 trajectory
Total Reward: -599.0253699412792 At 13 trajectory
Total Reward: -405.94222369645456 At 14 trajectory
Total Reward: -268.399682377904 At 15 trajectory
Total Reward: -118.16980465748576 At 16 trajectory
Total Reward: -101.49273476448545 At 17 trajectory
Total Reward: -266.1949331666435 At 18 trajectory
Total Reward: -204.66467076670915 At 19 trajectory
Total Reward: -171.9206229243997 At 20 trajectory
Total Reward: -166.457894169455 At 21 trajectory
Total Reward: -64.2066184271959 At 22 trajectory
Total Reward: -39.9611864648993 At 23 trajectory
Total Reward: -42.10067375891101 At 24 trajectory
Total Reward: -62.79187536806025 At 25 trajectory
Total Reward: -168.83025850013286 At 26 trajectory
Total Reward: -141.3351558515547 At 27 trajectory

[-214.6634730397425,
 -224.0199126058922,
 -26.703534196428393,
 -213.45580466429686,
 -185.8582124798869,
 -218.79906020355708,
 -81.23557539993581,
 -403.638325224108,
 -163.4153518452136,
 -511.6086220899163,
 -603.0642535598336,
 -468.73492592874254,
 -366.28771529484425,
 -599.0253699412792,
 -405.94222369645456,
 -268.399682377904,
 -118.16980465748576,
 -101.49273476448545,
 -266.1949331666435,
 -204.66467076670915,
 -171.9206229243997,
 -166.457894169455,
 -64.2066184271959,
 -39.9611864648993,
 -42.10067375891101,
 -62.79187536806025,
 -168.83025850013286,
 -141.3351558515547,
 -132.35598402526725,
 -135.29694809409818,
 -136.39387816456775,
 -127.66092044335181,
 -123.15095550963164,
 -105.69611576648391,
 -120.6814329432161,
 -117.33209199096815,
 -130.71652051038762,
 -166.577486195571,
 -164.46017684934003,
 -158.69285366191966,
 -230.44592790745597,
 -765.4164925589117,
 -910.3385751680754,
 -376.2063150003287,
 -229.26775344401761,
 -208.6120370938034,
 -199.069813605942